# Model Improvement

In [46]:
import pandas as pd
import numpy as np
import joblib
import time
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold,  cross_validate
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [3]:
X_train = joblib.load("../models/X_train_engineered_processed.pkl")
X_test = joblib.load("../models/X_test_engineered_processed.pkl")
y_train = joblib.load("../models/y_train.pkl")
y_test = joblib.load("../models/y_test.pkl")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 75)
X_test : (1409, 75)
y_train: (5634,)
y_test : (1409,)


In [9]:
baseline_results = pd.read_csv("../models/phase5_baseline_model_results.csv")
baseline_results

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,Training Time (sec)
0,XGBoost,0.756565,0.527337,0.799465,0.635494,0.846594,0.660382,1.026624
1,Random Forest,0.757275,0.529630,0.764706,0.625821,0.838903,0.643058,1.699015
2,Logistic Regression,0.736693,0.502564,0.786096,0.613139,0.840378,0.629892,0.461660
3,Decision Tree,0.726757,0.490909,0.794118,0.606742,0.821822,0.614773,0.137753
4,Gradient Boosting,0.804826,0.670103,0.521390,0.586466,0.844385,0.656003,5.346065
5,KNN,0.767921,0.568513,0.521390,0.543933,0.791043,0.531229,0.027988


In [10]:
top_3_models = (baseline_results.sort_values("F1-Score", ascending=False).head(3))
top_3_models

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,Training Time (sec)
0,XGBoost,0.756565,0.527337,0.799465,0.635494,0.846594,0.660382,1.026624
1,Random Forest,0.757275,0.529630,0.764706,0.625821,0.838903,0.643058,1.699015
2,Logistic Regression,0.736693,0.502564,0.786096,0.613139,0.840378,0.629892,0.461660


In [11]:
y_train_binary = (y_train == "Yes").astype(int)
print(y_train_binary.value_counts())

Churn
0    4139
1    1495
Name: count, dtype: int64


In [50]:
y_test_binary = (y_test == "Yes").astype(int)

#### XGBoost

In [13]:
scale_pos_weight = ((y_train_binary == 0).sum()/(y_train_binary == 1).sum())
xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

#### Random Forest

In [14]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

#### Logistic Regression

In [15]:
lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

## Cross Validation

In [16]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [17]:
def cross_validate_model(model_name, model, X, y, cv):

    scoring = {
        "Accuracy": "accuracy",
        "Precision": "precision",
        "Recall": "recall",
        "F1-Score": "f1",
        "ROC-AUC": "roc_auc",
        "PR-AUC": "average_precision"
    }

    results = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    return {
        "Model": model_name,

        "Accuracy": results["test_Accuracy"].mean(),
        "Precision": results["test_Precision"].mean(),
        "Recall": results["test_Recall"].mean(),
        "F1-Score": results["test_F1-Score"].mean(),
        "ROC-AUC": results["test_ROC-AUC"].mean(),
        "PR-AUC": results["test_PR-AUC"].mean(),

        "Accuracy Std": results["test_Accuracy"].std(),
        "Precision Std": results["test_Precision"].std(),
        "Recall Std": results["test_Recall"].std(),
        "F1 Std": results["test_F1-Score"].std(),
        "ROC-AUC Std": results["test_ROC-AUC"].std(),
        "PR-AUC Std": results["test_PR-AUC"].std(),

        "Mean Fit Time": results["fit_time"].mean(),
        "Mean Score Time": results["score_time"].mean()
    }

In [18]:
xgb_cv_results = cross_validate_model(
    "XGBoost",
    xgb_model,
    X_train,
    y_train_binary,
    cv
)

xgb_cv_results

{'Model': 'XGBoost',
 'Accuracy': np.float64(0.7566575939202618),
 'Precision': np.float64(0.527704292926462),
 'Recall': np.float64(0.7913043478260869),
 'F1-Score': np.float64(0.6330590258347918),
 'ROC-AUC': np.float64(0.8481883125089833),
 'PR-AUC': np.float64(0.667882745071041),
 'Accuracy Std': np.float64(0.009785086439926005),
 'Precision Std': np.float64(0.012449646931461543),
 'Recall Std': np.float64(0.02561095614592196),
 'F1 Std': np.float64(0.015433971214009194),
 'ROC-AUC Std': np.float64(0.011571639554228411),
 'PR-AUC Std': np.float64(0.023419670562506663),
 'Mean Fit Time': np.float64(0.9824522972106934),
 'Mean Score Time': np.float64(0.13538150787353515)}

In [19]:
rf_cv_results = cross_validate_model(
    "Random Forest",
    rf_model,
    X_train,
    y_train_binary,
    cv
)

rf_cv_results

{'Model': 'Random Forest',
 'Accuracy': np.float64(0.7657095891101827),
 'Precision': np.float64(0.5419774226316371),
 'Recall': np.float64(0.7578595317725751),
 'F1-Score': np.float64(0.6318818803974648),
 'ROC-AUC': np.float64(0.8447250370324524),
 'PR-AUC': np.float64(0.6626035578215053),
 'Accuracy Std': np.float64(0.00742340530041553),
 'Precision Std': np.float64(0.010595189617993436),
 'Recall Std': np.float64(0.018633295354092503),
 'F1 Std': np.float64(0.010969156076387028),
 'ROC-AUC Std': np.float64(0.009579716364755502),
 'PR-AUC Std': np.float64(0.018636782490358925),
 'Mean Fit Time': np.float64(1.4538341522216798),
 'Mean Score Time': np.float64(0.265259838104248)}

In [20]:
lr_cv_results = cross_validate_model(
    "Logistic Regression",
    lr_model,
    X_train,
    y_train_binary,
    cv
)

lr_cv_results

{'Model': 'Logistic Regression',
 'Accuracy': np.float64(0.7502687939026101),
 'Precision': np.float64(0.5192270237974526),
 'Recall': np.float64(0.797324414715719),
 'F1-Score': np.float64(0.6287701888320376),
 'ROC-AUC': np.float64(0.8451469752836891),
 'PR-AUC': np.float64(0.6577878942597526),
 'Accuracy Std': np.float64(0.014026959657319763),
 'Precision Std': np.float64(0.017497082296506566),
 'Recall Std': np.float64(0.033618279259208204),
 'F1 Std': np.float64(0.02143002896400556),
 'ROC-AUC Std': np.float64(0.011622715295879672),
 'PR-AUC Std': np.float64(0.01557772245083647),
 'Mean Fit Time': np.float64(0.20596137046813964),
 'Mean Score Time': np.float64(0.039787817001342776)}

In [21]:
cv_results = pd.DataFrame([
    xgb_cv_results,
    rf_cv_results,
    lr_cv_results
])

In [22]:
cv_results = cv_results.sort_values(
    "F1-Score",
    ascending=False
).reset_index(drop=True)

cv_results

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,Accuracy Std,Precision Std,Recall Std,F1 Std,ROC-AUC Std,PR-AUC Std,Mean Fit Time,Mean Score Time
0,XGBoost,0.756658,0.527704,0.791304,0.633059,0.848188,0.667883,0.009785,0.012450,0.025611,0.015434,0.011572,0.023420,0.982452,0.135382
1,Random Forest,0.765710,0.541977,0.757860,0.631882,0.844725,0.662604,0.007423,0.010595,0.018633,0.010969,0.009580,0.018637,1.453834,0.265260
2,Logistic Regression,0.750269,0.519227,0.797324,0.628770,0.845147,0.657788,0.014027,0.017497,0.033618,0.021430,0.011623,0.015578,0.205961,0.039788


In [23]:
cv_results.to_csv("../models/phase11_cross_validation_results.csv",index=False)
print("Cross-validation results saved successfully.")

Cross-validation results saved successfully.


#### Class Weighting

In [24]:
from sklearn.utils.class_weight import compute_class_weight
classes = np.array([0, 1])
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_binary
)
class_weight_dict = dict(
    zip(classes, class_weights)
)
print("Class weights:", class_weight_dict)

Class weights: {np.int64(0): np.float64(0.6805991785455424), np.int64(1): np.float64(1.8842809364548494)}


#### XGBoost

In [25]:
xgb_weighted = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=class_weight_dict[1] / class_weight_dict[0],
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

#### Random Forest

In [26]:
rf_weighted = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight=class_weight_dict,
    random_state=42,
    n_jobs=-1
)

#### Logistic Regression

In [27]:
lr_weighted = LogisticRegression(
    class_weight=class_weight_dict,
    max_iter=1000,
    random_state=42
)

In [28]:
xgb_weighted_results = cross_validate_model(
    "XGBoost - Weighted",
    xgb_weighted,
    X_train,
    y_train_binary,
    cv
)

rf_weighted_results = cross_validate_model(
    "Random Forest - Weighted",
    rf_weighted,
    X_train,
    y_train_binary,
    cv
)

lr_weighted_results = cross_validate_model(
    "Logistic Regression - Weighted",
    lr_weighted,
    X_train,
    y_train_binary,
    cv
)

In [29]:
class_weight_results = pd.DataFrame([
    xgb_cv_results,
    xgb_weighted_results,
    rf_cv_results,
    rf_weighted_results,
    lr_cv_results,
    lr_weighted_results
])

class_weight_results = class_weight_results.sort_values(
    "F1-Score",
    ascending=False
).reset_index(drop=True)

class_weight_results

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,Accuracy Std,Precision Std,Recall Std,F1 Std,ROC-AUC Std,PR-AUC Std,Mean Fit Time,Mean Score Time
0,XGBoost,0.756658,0.527704,0.791304,0.633059,0.848188,0.667883,0.009785,0.012450,0.025611,0.015434,0.011572,0.023420,0.982452,0.135382
1,XGBoost - Weighted,0.756658,0.527704,0.791304,0.633059,0.848188,0.667883,0.009785,0.012450,0.025611,0.015434,0.011572,0.023420,0.886524,0.065386
2,Random Forest,0.765710,0.541977,0.757860,0.631882,0.844725,0.662604,0.007423,0.010595,0.018633,0.010969,0.009580,0.018637,1.453834,0.265260
3,Random Forest - Weighted,0.763225,0.538328,0.758528,0.629656,0.844470,0.661161,0.008402,0.011675,0.017366,0.012121,0.009315,0.020466,1.633700,0.383227
4,Logistic Regression - Weighted,0.750979,0.520122,0.797993,0.629645,0.845147,0.657631,0.014458,0.018037,0.033418,0.021892,0.011639,0.015598,0.332164,0.044698
5,Logistic Regression,0.750269,0.519227,0.797324,0.628770,0.845147,0.657788,0.014027,0.017497,0.033618,0.021430,0.011623,0.015578,0.205961,0.039788


In [30]:
best_class_weight = {}

for model_name in [
    "XGBoost",
    "Random Forest",
    "Logistic Regression"
]:
    
    model_rows = class_weight_results[
        class_weight_results["Model"].str.startswith(model_name)
    ]

    best_row = model_rows.loc[
        model_rows["F1-Score"].idxmax()
    ]

    best_class_weight[model_name] = best_row

best_class_weight_df = pd.DataFrame(
    best_class_weight
).T

best_class_weight_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,Accuracy Std,Precision Std,Recall Std,F1 Std,ROC-AUC Std,PR-AUC Std,Mean Fit Time,Mean Score Time
XGBoost,XGBoost,0.756658,0.527704,0.791304,0.633059,0.848188,0.667883,0.009785,0.01245,0.025611,0.015434,0.011572,0.02342,0.982452,0.135382
Random Forest,Random Forest,0.76571,0.541977,0.75786,0.631882,0.844725,0.662604,0.007423,0.010595,0.018633,0.010969,0.00958,0.018637,1.453834,0.26526
Logistic Regression,Logistic Regression - Weighted,0.750979,0.520122,0.797993,0.629645,0.845147,0.657631,0.014458,0.018037,0.033418,0.021892,0.011639,0.015598,0.332164,0.044698


In [31]:
class_weight_results.to_csv(
    "../models/phase11_class_weighting_results.csv",
    index=False
)

best_class_weight_df.to_csv(
    "../models/phase11_best_class_weighting_models.csv",
    index=False
)

print("Class-weighting results saved successfully.")

Class-weighting results saved successfully.


#### Hyperparameter tuning

#### XGBoost

In [32]:
xgb_tuning_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb_param_grid = {
    "n_estimators": [100, 150, 200],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0]
}

In [33]:
xgb_search = RandomizedSearchCV(
    estimator=xgb_tuning_model,
    param_distributions=xgb_param_grid,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(
    X_train,
    y_train_binary
)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.8, 0.9, ...], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [3, 4, ...], 'n_estimators': [100, 150, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` c

In [34]:
print("XGBoost Best Parameters:")
print(xgb_search.best_params_)

print("\nXGBoost Best CV F1:")
print(xgb_search.best_score_)

XGBoost Best Parameters:
{'subsample': 0.8, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.9}

XGBoost Best CV F1:
0.6370251405803844


#### Random Forest

In [35]:
rf_tuning_model = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_param_grid = {
    "n_estimators": [100, 150, 200, 250],
    "max_depth": [None, 8, 10, 12, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [36]:
rf_search = RandomizedSearchCV(
    estimator=rf_tuning_model,
    param_distributions=rf_param_grid,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(
    X_train,
    y_train_binary
)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [None, 8, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can b

In [37]:
print("Random Forest Best Parameters:")
print(rf_search.best_params_)

print("\nRandom Forest Best CV F1:")
print(rf_search.best_score_)

Random Forest Best Parameters:
{'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None}

Random Forest Best CV F1:
0.6344094606158321


#### Logistic Regression

In [38]:
lr_tuning_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

lr_param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"],
}

In [39]:
lr_search = RandomizedSearchCV(
    estimator=lr_tuning_model,
    param_distributions=lr_param_grid,
    n_iter=8,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

lr_search.fit(
    X_train,
    y_train_binary
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'C': [0.01, 0.1, ...], 'solver': ['liblinear', 'lbfgs']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",8
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_re

In [40]:
print("Logistic Regression Best Parameters:")
print(lr_search.best_params_)

print("\nLogistic Regression Best CV F1:")
print(lr_search.best_score_)

Logistic Regression Best Parameters:
{'solver': 'liblinear', 'C': 0.01}

Logistic Regression Best CV F1:
0.6300112905806571


In [41]:
best_xgb = xgb_search.best_estimator_
best_rf = rf_search.best_estimator_
best_lr = lr_search.best_estimator_

In [42]:
xgb_tuned_results = cross_validate_model(
    "XGBoost - Tuned",
    best_xgb,
    X_train,
    y_train_binary,
    cv
)

rf_tuned_results = cross_validate_model(
    "Random Forest - Tuned",
    best_rf,
    X_train,
    y_train_binary,
    cv
)

lr_tuned_results = cross_validate_model(
    "Logistic Regression - Tuned",
    best_lr,
    X_train,
    y_train_binary,
    cv
)

In [43]:
tuning_results = pd.DataFrame([
    xgb_tuned_results,
    rf_tuned_results,
    lr_tuned_results
])

tuning_results = tuning_results.sort_values(
    "F1-Score",
    ascending=False
).reset_index(drop=True)

tuning_results

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,Accuracy Std,Precision Std,Recall Std,F1 Std,ROC-AUC Std,PR-AUC Std,Mean Fit Time,Mean Score Time
0,XGBoost - Tuned,0.758256,0.529675,0.799331,0.637025,0.849202,0.670464,0.011805,0.015159,0.022782,0.016032,0.011122,0.019518,0.590709,0.059613
1,Random Forest - Tuned,0.767662,0.544705,0.759866,0.634409,0.843384,0.652359,0.006360,0.009210,0.020000,0.010019,0.010012,0.025653,1.159074,0.259478
2,Logistic Regression - Tuned,0.752398,0.522020,0.794649,0.630011,0.845641,0.657124,0.014132,0.017720,0.031841,0.021497,0.010541,0.012239,0.212701,0.037977


In [44]:
best_parameters = pd.DataFrame({
    "Model": [
        "XGBoost",
        "Random Forest",
        "Logistic Regression"
    ],
    "Best Parameters": [
        xgb_search.best_params_,
        rf_search.best_params_,
        lr_search.best_params_
    ]
})

best_parameters

,Model,Best Parameters
0,XGBoost,"{'subsample': 0.8, 'n_estimators': 100, 'max_d..."
1,Random Forest,"{'n_estimators': 100, 'min_samples_split': 10,..."
2,Logistic Regression,"{'solver': 'liblinear', 'C': 0.01}"


In [45]:
best_parameters.to_csv(
    "../models/phase11_best_hyperparameters.csv",
    index=False
)

tuning_results.to_csv(
    "../models/phase11_hyperparameter_tuning_results.csv",
    index=False
)

print("Hyperparameter tuning results saved.")

Hyperparameter tuning results saved.


#### Feature Selection

In [47]:
k_values = [10, 20, 30, 40, 50]

In [58]:
from sklearn.base import clone
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


def evaluate_feature_selection(
    model_name,
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    k_values
):
    
    results = []

    for k in k_values:

        # Make sure K is not larger than total features
        k_use = min(k, X_train.shape[1])

        # Create feature selector
        selector = SelectKBest(
            score_func=f_classif,
            k=k_use
        )

        # Fit selector only on training data
        X_train_selected = selector.fit_transform(
            X_train,
            y_train
        )

        # Apply same selector to test data
        X_test_selected = selector.transform(
            X_test
        )

        # Create a fresh copy of the model
        feature_model = clone(model)

        # Train model
        feature_model.fit(
            X_train_selected,
            y_train
        )

        # Predictions
        y_pred = feature_model.predict(
            X_test_selected
        )

        # Prediction probabilities
        y_proba = feature_model.predict_proba(
            X_test_selected
        )[:, 1]

        # Evaluation metrics
        results.append({
            "Model": model_name,
            "K": k_use,

            "Accuracy": accuracy_score(
                y_test,
                y_pred
            ),

            "Precision": precision_score(
                y_test,
                y_pred,
                zero_division=0
            ),

            "Recall": recall_score(
                y_test,
                y_pred,
                zero_division=0
            ),

            "F1-Score": f1_score(
                y_test,
                y_pred,
                zero_division=0
            ),

            "ROC-AUC": roc_auc_score(
                y_test,
                y_proba
            ),

            "PR-AUC": average_precision_score(
                y_test,
                y_proba
            )
        })

    return pd.DataFrame(results)

#### XGBoost

In [59]:
xgb_feature_results = evaluate_feature_selection(
    "XGBoost",
    best_xgb,
    X_train,
    y_train_binary,
    X_test,
    y_test_binary,
    k_values
)

xgb_feature_results

,Model,K,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
0,XGBoost,10,0.743080,0.510274,0.796791,0.622129,0.837640,0.642273
1,XGBoost,20,0.748048,0.516408,0.799465,0.627492,0.839881,0.650578
2,XGBoost,30,0.751597,0.520906,0.799465,0.630802,0.844350,0.663304
3,XGBoost,40,0.747339,0.515464,0.802139,0.627615,0.844184,0.664689
4,XGBoost,50,0.753016,0.522569,0.804813,0.633684,0.844625,0.663665


#### Random Forest

In [60]:
rf_feature_results = evaluate_feature_selection(
    "Random Forest",
    best_rf,
    X_train,
    y_train_binary,
    X_test,
    y_test_binary,
    k_values
)

rf_feature_results

,Model,K,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
0,Random Forest,10,0.738822,0.505155,0.786096,0.615063,0.822416,0.612858
1,Random Forest,20,0.743080,0.511583,0.708556,0.594170,0.819263,0.596073
2,Random Forest,30,0.754436,0.527559,0.716578,0.607710,0.830563,0.631450
3,Random Forest,40,0.760114,0.534884,0.737968,0.620225,0.831502,0.629219
4,Random Forest,50,0.766501,0.542694,0.764706,0.634850,0.837770,0.636881


#### Logistic Regression


In [61]:
lr_feature_results = evaluate_feature_selection(
    "Logistic Regression",
    best_lr,
    X_train,
    y_train_binary,
    X_test,
    y_test_binary,
    k_values
)

lr_feature_results

,Model,K,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
0,Logistic Regression,10,0.735983,0.501701,0.788770,0.613306,0.833602,0.635593
1,Logistic Regression,20,0.740951,0.507692,0.794118,0.619395,0.835016,0.628833
2,Logistic Regression,30,0.742370,0.509338,0.802139,0.623053,0.835750,0.625794
3,Logistic Regression,40,0.745919,0.513889,0.791444,0.623158,0.836222,0.627730
4,Logistic Regression,50,0.749468,0.518198,0.799465,0.628812,0.838180,0.624916


In [62]:
feature_selection_results = pd.concat(
    [
        xgb_feature_results,
        rf_feature_results,
        lr_feature_results
    ],
    ignore_index=True
)

feature_selection_results

,Model,K,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
0,XGBoost,10,0.743080,0.510274,0.796791,0.622129,0.837640,0.642273
1,XGBoost,20,0.748048,0.516408,0.799465,0.627492,0.839881,0.650578
2,XGBoost,30,0.751597,0.520906,0.799465,0.630802,0.844350,0.663304
3,XGBoost,40,0.747339,0.515464,0.802139,0.627615,0.844184,0.664689
4,XGBoost,50,0.753016,0.522569,0.804813,0.633684,0.844625,0.663665
5,Random Forest,10,0.738822,0.505155,0.786096,0.615063,0.822416,0.612858
6,Random Forest,20,0.743080,0.511583,0.708556,0.594170,0.819263,0.596073
7,Random Forest,30,0.754436,0.527559,0.716578,0.607710,0.830563,0.631450
8,Random Forest,40,0.760114,0.534884,0.737968,0.620225,0.831502,0.629219
9,Random Forest,50,0.766501,0.542694,0.764706,0.634850,0.837770,0.636881


In [63]:
feature_selection_results = (
    feature_selection_results
    .sort_values(
        "F1-Score",
        ascending=False
    )
    .reset_index(drop=True)
)

feature_selection_results

,Model,K,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
0,Random Forest,50,0.766501,0.542694,0.764706,0.634850,0.837770,0.636881
1,XGBoost,50,0.753016,0.522569,0.804813,0.633684,0.844625,0.663665
2,XGBoost,30,0.751597,0.520906,0.799465,0.630802,0.844350,0.663304
3,Logistic Regression,50,0.749468,0.518198,0.799465,0.628812,0.838180,0.624916
4,XGBoost,40,0.747339,0.515464,0.802139,0.627615,0.844184,0.664689
5,XGBoost,20,0.748048,0.516408,0.799465,0.627492,0.839881,0.650578
6,Logistic Regression,40,0.745919,0.513889,0.791444,0.623158,0.836222,0.627730
7,Logistic Regression,30,0.742370,0.509338,0.802139,0.623053,0.835750,0.625794
8,XGBoost,10,0.743080,0.510274,0.796791,0.622129,0.837640,0.642273
9,Random Forest,40,0.760114,0.534884,0.737968,0.620225,0.831502,0.629219


In [64]:
best_k_results = (
    feature_selection_results
    .loc[
        feature_selection_results
        .groupby("Model")["F1-Score"]
        .idxmax()
    ]
    .reset_index(drop=True)
)

best_k_results

,Model,K,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
0,Logistic Regression,50,0.749468,0.518198,0.799465,0.628812,0.838180,0.624916
1,Random Forest,50,0.766501,0.542694,0.764706,0.634850,0.837770,0.636881
2,XGBoost,50,0.753016,0.522569,0.804813,0.633684,0.844625,0.663665


#### threshold function

In [65]:
from sklearn.model_selection import train_test_split

X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train,
    y_train_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_train_binary
)

print("Training:", X_train_fit.shape)
print("Validation:", X_val.shape)

Training: (4507, 75)
Validation: (1127, 75)


In [66]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def find_best_threshold(model, X_val, y_val):

    y_proba = model.predict_proba(X_val)[:, 1]

    thresholds = np.arange(0.30, 0.71, 0.05)

    results = []

    for threshold in thresholds:

        y_pred = (
            y_proba >= threshold
        ).astype(int)

        results.append({
            "Threshold": round(threshold, 2),
            "Accuracy": accuracy_score(y_val, y_pred),
            "Precision": precision_score(
                y_val, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_val, y_pred, zero_division=0
            ),
            "F1-Score": f1_score(
                y_val, y_pred, zero_division=0
            )
        })

    results_df = pd.DataFrame(results)

    best_row = results_df.loc[
        results_df["F1-Score"].idxmax()
    ]

    return results_df, best_row

In [67]:
xgb_final_model = XGBClassifier(
    **xgb_search.best_params_,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_final_model.fit(
    X_train_fit,
    y_train_fit
)

xgb_threshold_results, xgb_best_threshold = find_best_threshold(
    xgb_final_model,
    X_val,
    y_val
)

xgb_threshold_results

,Threshold,Accuracy,Precision,Recall,F1-Score
0,0.30,0.648625,0.423381,0.896321,0.575107
1,0.35,0.676131,0.443493,0.866221,0.586636
2,0.40,0.709849,0.473485,0.836120,0.604595
3,0.45,0.735581,0.501031,0.812709,0.619898
4,0.50,0.762201,0.535469,0.782609,0.635870
5,0.55,0.775510,0.557500,0.745819,0.638054
6,0.60,0.794144,0.593838,0.709030,0.646341
7,0.65,0.814552,0.652027,0.645485,0.648739
8,0.70,0.816327,0.684000,0.571906,0.622951


In [68]:
print("XGBoost best threshold:")
print(xgb_best_threshold)

XGBoost best threshold:
Threshold    0.650000
Accuracy     0.814552
Precision    0.652027
Recall       0.645485
F1-Score     0.648739
Name: 7, dtype: float64


In [69]:
rf_final_model = RandomForestClassifier(
    **rf_search.best_params_,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_final_model.fit(
    X_train_fit,
    y_train_fit
)

rf_threshold_results, rf_best_threshold = find_best_threshold(
    rf_final_model,
    X_val,
    y_val
)

rf_threshold_results

,Threshold,Accuracy,Precision,Recall,F1-Score
0,0.30,0.677019,0.446103,0.899666,0.596452
1,0.35,0.708962,0.473779,0.876254,0.615023
2,0.40,0.726708,0.491124,0.832776,0.617866
3,0.45,0.749778,0.518764,0.785953,0.625000
4,0.50,0.769299,0.548148,0.742475,0.630682
5,0.55,0.785271,0.577236,0.712375,0.637725
6,0.60,0.793256,0.603125,0.645485,0.623586
7,0.65,0.801242,0.644788,0.558528,0.598566
8,0.70,0.808341,0.694836,0.494983,0.578125


In [70]:
print("Random Forest best threshold:")
print(rf_best_threshold)

Random Forest best threshold:
Threshold    0.550000
Accuracy     0.785271
Precision    0.577236
Recall       0.712375
F1-Score     0.637725
Name: 5, dtype: float64


In [71]:
lr_final_model = LogisticRegression(
    **lr_search.best_params_,
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

lr_final_model.fit(
    X_train_fit,
    y_train_fit
)

lr_threshold_results, lr_best_threshold = find_best_threshold(
    lr_final_model,
    X_val,
    y_val
)

lr_threshold_results

,Threshold,Accuracy,Precision,Recall,F1-Score
0,0.30,0.638864,0.417933,0.919732,0.574713
1,0.35,0.665484,0.435855,0.886288,0.584344
2,0.40,0.703638,0.467412,0.839465,0.600478
3,0.45,0.728483,0.492901,0.812709,0.613636
4,0.50,0.755989,0.526906,0.785953,0.630872
5,0.55,0.772848,0.552826,0.752508,0.637394
6,0.60,0.790594,0.586301,0.715719,0.644578
7,0.65,0.808341,0.636964,0.645485,0.641196
8,0.70,0.818988,0.690763,0.575251,0.627737


In [72]:
print("Logistic Regression best threshold:")
print(lr_best_threshold)

Logistic Regression best threshold:
Threshold    0.600000
Accuracy     0.790594
Precision    0.586301
Recall       0.715719
F1-Score     0.644578
Name: 6, dtype: float64


In [73]:
threshold_results = pd.DataFrame([
    {
        "Model": "XGBoost",
        "Best Threshold": xgb_best_threshold["Threshold"],
        "Accuracy": xgb_best_threshold["Accuracy"],
        "Precision": xgb_best_threshold["Precision"],
        "Recall": xgb_best_threshold["Recall"],
        "F1-Score": xgb_best_threshold["F1-Score"]
    },
    {
        "Model": "Random Forest",
        "Best Threshold": rf_best_threshold["Threshold"],
        "Accuracy": rf_best_threshold["Accuracy"],
        "Precision": rf_best_threshold["Precision"],
        "Recall": rf_best_threshold["Recall"],
        "F1-Score": rf_best_threshold["F1-Score"]
    },
    {
        "Model": "Logistic Regression",
        "Best Threshold": lr_best_threshold["Threshold"],
        "Accuracy": lr_best_threshold["Accuracy"],
        "Precision": lr_best_threshold["Precision"],
        "Recall": lr_best_threshold["Recall"],
        "F1-Score": lr_best_threshold["F1-Score"]
    }
])

threshold_results = threshold_results.sort_values(
    "F1-Score",
    ascending=False
).reset_index(drop=True)

threshold_results

,Model,Best Threshold,Accuracy,Precision,Recall,F1-Score
0,XGBoost,0.65,0.814552,0.652027,0.645485,0.648739
1,Logistic Regression,0.60,0.790594,0.586301,0.715719,0.644578
2,Random Forest,0.55,0.785271,0.577236,0.712375,0.637725


In [74]:
final_test_results = []

final_models = [
    ("XGBoost", xgb_final_model, xgb_best_threshold["Threshold"]),
    ("Random Forest", rf_final_model, rf_best_threshold["Threshold"]),
    ("Logistic Regression", lr_final_model, lr_best_threshold["Threshold"])
]

for model_name, model, threshold in final_models:

    y_proba = model.predict_proba(X_test)[:, 1]

    y_pred = (
        y_proba >= threshold
    ).astype(int)

    final_test_results.append({
        "Model": model_name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test_binary, y_pred),
        "Precision": precision_score(
            y_test_binary, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test_binary, y_pred, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test_binary, y_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test_binary, y_proba
        ),
        "PR-AUC": average_precision_score(
            y_test_binary, y_proba
        )
    })

final_test_results = pd.DataFrame(final_test_results)

final_test_results = final_test_results.sort_values(
    "F1-Score",
    ascending=False
).reset_index(drop=True)

final_test_results

,Model,Threshold,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
0,XGBoost,0.65,0.797019,0.611111,0.647059,0.628571,0.843167,0.656205
1,Logistic Regression,0.60,0.774308,0.559322,0.705882,0.624113,0.841040,0.628601
2,Random Forest,0.55,0.772179,0.555556,0.708556,0.622797,0.837291,0.630384


In [75]:
final_test_results.to_csv(
    "../models/phase11_final_improved_model_results.csv",
    index=False
)

threshold_results.to_csv(
    "../models/phase11_threshold_tuning_results.csv",
    index=False
)

print("Final model improvement results saved.")

Final model improvement results saved.


In [76]:
best_model_name = final_test_results.loc[
    final_test_results["F1-Score"].idxmax(),
    "Model"
]

print("Best final model:", best_model_name)

Best final model: XGBoost


In [77]:
best_threshold = final_test_results.loc[
    final_test_results["Model"] == best_model_name,
    "Threshold"
].iloc[0]

print("Best threshold:", best_threshold)

Best threshold: 0.65


In [78]:
final_models_dict = {
    "XGBoost": xgb_final_model,
    "Random Forest": rf_final_model,
    "Logistic Regression": lr_final_model
}

In [79]:
final_model = final_models_dict[best_model_name]

print("Selected model:", best_model_name)

Selected model: XGBoost


In [80]:
import joblib

joblib.dump(
    final_model,
    "../models/final_churn_model.pkl"
)

print("Final model saved successfully.")

Final model saved successfully.


In [81]:
joblib.dump(
    best_threshold,
    "../models/final_churn_threshold.pkl"
)

print("Final threshold saved successfully.")

Final threshold saved successfully.


In [82]:
print("Final Model:", best_model_name)
print("Decision Threshold:", best_threshold)

Final Model: XGBoost
Decision Threshold: 0.65
